# Python + NumPy for ML: The Foundation

Before you can build RAG, agents, or train models, you need to be fluent in NumPy. This notebook covers the NumPy patterns that appear constantly in ML code — not NumPy in general, but specifically the operations you'll use when working with embeddings, matrices, and similarity search.

**4 exercises.** Each one is 1–5 lines of code. The goal isn't to write a lot — it's to understand exactly what those lines are doing.

### What you'll learn
| Concept | Why it matters in ML |
|---------|---------------------|
| Shape manipulation | Embeddings, batches, and weight matrices are all defined by shape |
| Broadcasting | Subtract a mean from a whole batch in one line — no loop |
| Matrix multiply | The backbone of attention, linear layers, and similarity search |
| Normalization | Most embedding models output unit-norm vectors — this is why |
| Top-k selection | The core of every vector search / retrieval system |

### Prerequisites
```bash
pip install numpy matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(f"NumPy version: {np.__version__}")

---
## Part 1: Why NumPy?

ML deals with large arrays of numbers — embeddings with 768 dimensions, weight matrices with millions of parameters, batches of 1024 vectors at a time. Python loops over these are catastrophically slow.

NumPy solves this by calling optimized C and Fortran routines under the hood, running vectorized operations on entire arrays at once. The speedup is not marginal — it's typically 100x or more.

Let's measure it.

In [ ]:
N = 1_000_000
a_list = list(range(N))
b_list = list(range(N))
a_arr  = np.arange(N, dtype=np.float64)
b_arr  = np.arange(N, dtype=np.float64)

# Python loop dot product
t0 = time.perf_counter()
python_result = sum(x * y for x, y in zip(a_list, b_list))
python_time = time.perf_counter() - t0

# NumPy dot product
t0 = time.perf_counter()
numpy_result = np.dot(a_arr, b_arr)
numpy_time = time.perf_counter() - t0

speedup = python_time / numpy_time
print(f"Python loop: {python_time * 1000:.1f} ms")
print(f"NumPy dot:   {numpy_time * 1000:.2f} ms")
print(f"Speedup:     {speedup:.0f}x")
print()
print("Same result?")
print(f"  Python: {python_result:.0f}")
print(f"  NumPy:  {numpy_result:.0f}")
print()
print("In ML you run operations like this millions of times per training step.")
print("NumPy (or PyTorch/JAX, which share the same ideas) is non-negotiable.")

---
## Part 2: Arrays and Shape

**Shape is the most important concept in NumPy for ML.** Every time something breaks, it's usually a shape mismatch.

Here's how shapes map to ML objects:

| Shape | What it represents |
|-------|-------------------|
| `(384,)` | One embedding vector (384-dimensional) |
| `(10, 384)` | A batch of 10 embeddings |
| `(384, 768)` | A weight matrix — maps 384-dim input to 768-dim output |
| `(32, 10, 64)` | A batch of 32 attention outputs, 10 heads, 64-dim each |

Get comfortable reading shapes out loud: "32 batches of 10 vectors each with 64 dimensions."

In [ ]:
# Creating arrays
zeros = np.zeros((10, 384))        # a batch of 10 zero-embeddings
ones  = np.ones((5, 384))          # a batch of 5 all-ones embeddings
random = np.random.randn(10, 384)  # simulated batch of 10 real embeddings (Gaussian noise)
from_list = np.array([1.0, 2.0, 3.0, 4.0])  # from a Python list

print("zeros.shape:",   zeros.shape,  "  dtype:", zeros.dtype,  "  ndim:", zeros.ndim)
print("ones.shape:",    ones.shape,   "  dtype:", ones.dtype,   "  ndim:", ones.ndim)
print("random.shape:",  random.shape, "  dtype:", random.dtype, "  ndim:", random.ndim)
print("from_list.shape:", from_list.shape)
print()

# Reshape: same data, different view
flat  = np.random.randn(384)     # shape (384,) — one embedding
batch = flat.reshape(1, 384)     # shape (1, 384) — a "batch" of 1
print("flat.shape:",  flat.shape)
print("batch.shape:", batch.shape)
print()
print("Same data?")
print("  flat[0]  =", flat[0])
print("  batch[0, 0] =", batch[0, 0])

### Exercise 1: Reshape a flat array into a batch

Imagine you've read raw data off disk — pixel values, or concatenated embedding segments — as a flat 1D array. Before you can pass it to any ML model you need to reshape it into the expected batch format.

**Your task:** Given a flat array of 1920 floats, reshape it into a batch of 5 embeddings of dimension 384. Then verify the shape.

**Hint:** `array.reshape(rows, cols)` — you need the product `rows * cols == 1920`.

In [ ]:
flat = np.random.randn(1920)

# YOUR TURN: reshape `flat` into a batch of 5 embeddings of dimension 384.
# One line.
batch = ???  # ← replace this
raise NotImplementedError("Replace ??? with flat.reshape(...)")

# ── Test ──────────────────────────────────────────────────────────────────────
assert batch.shape == (5, 384), f"Expected (5, 384), got {batch.shape}"
assert batch[0, 0] == flat[0], "Data should be the same — just a different view"
print(f"flat.shape:  {flat.shape}")
print(f"batch.shape: {batch.shape}")
print(f"✅ Passed! {batch.shape[0]} embeddings, each with {batch.shape[1]} dimensions.")

<details>
<summary>Solution (click to expand)</summary>

```python
batch = flat.reshape(5, 384)
```

Or equivalently: `flat.reshape(5, -1)` — the `-1` tells NumPy to infer that dimension automatically. This is common in real code because it's robust to changes in embedding dimension.

</details>

---
## Part 3: Indexing and Slicing

You'll constantly need to grab specific rows (documents), columns (features), or slices (sub-batches). NumPy's indexing syntax covers all of these.

In [ ]:
# A simulated batch: 8 embeddings of dimension 6 (tiny, so we can print it)
batch = np.round(np.random.randn(8, 6), 2)
print("Full batch (8 embeddings x 6 dims):")
print(batch)
print()

# Single element
print("batch[2, 4]    (row 2, col 4):", batch[2, 4])

# Entire row = one embedding
print("batch[3]       (embedding 3):", batch[3])

# Entire column = one feature across all embeddings
print("batch[:, 1]    (feature 1, all embeddings):", batch[:, 1])

# Slice of rows = a sub-batch
print("batch[2:5]     (embeddings 2, 3, 4):")
print(batch[2:5])

# Boolean mask — select rows where first feature is positive
mask = batch[:, 0] > 0
print(f"\nbatch[batch[:, 0] > 0]  ({mask.sum()} embeddings where feature 0 is positive):")
print(batch[mask])

In [ ]:
# Practical ML example: select the best embedding from a batch based on a score
# This pattern appears in beam search, top-k retrieval, and attention masking.

np.random.seed(42)
embeddings = np.random.randn(8, 384)   # 8 candidate embeddings
scores     = np.random.rand(8)          # one relevance score per embedding

print("Scores:", np.round(scores, 3))

# Get the embedding with the highest score
best_idx       = np.argmax(scores)
best_embedding = embeddings[best_idx]
print(f"\nBest embedding: index {best_idx} (score={scores[best_idx]:.3f})")
print(f"Shape: {best_embedding.shape}")

# Get the top-3 embeddings
top3_idx       = np.argsort(scores)[-3:][::-1]   # argsort ascending; flip to descending
top3_scores    = scores[top3_idx]
top3_embeddings = embeddings[top3_idx]
print(f"\nTop-3 indices: {top3_idx}")
print(f"Top-3 scores:  {np.round(top3_scores, 3)}")
print(f"Top-3 batch shape: {top3_embeddings.shape}")

---
## Part 4: Broadcasting

Broadcasting lets NumPy do operations on arrays of different shapes without copying data. It's the reason you can subtract a mean vector from an entire batch in one line — and it's why a lot of ML code looks deceptively short.

**The rule:** NumPy aligns shapes from the right. Dimensions must either match, or one of them must be 1 (which gets "stretched" to match the other).

```
(10, 384)   ←  a batch of 10 embeddings
    (384,)  ←  a single mean vector
----------
(10, 384)   ←  result: mean subtracted from each row
```

If the shapes don't align, you get a `ValueError`. Learning to read that error is a core ML debugging skill.

In [ ]:
np.random.seed(0)
batch = np.random.randn(10, 384)   # 10 embeddings, 384 dims each

# --- Example 1: subtract the mean vector (mean-centering a batch) ---
mean_vec = batch.mean(axis=0)      # shape (384,) — average across the 10 embeddings
centered = batch - mean_vec        # (10, 384) - (384,) → broadcasts to (10, 384)
print(f"batch.shape:    {batch.shape}")
print(f"mean_vec.shape: {mean_vec.shape}")
print(f"centered.shape: {centered.shape}")
print(f"Mean of centered batch (should be ~0): {centered.mean(axis=0)[:3].round(10)}")
print()

# --- Example 2: add a bias vector (what a linear layer does) ---
bias = np.array([0.1] * 384)       # shape (384,)
biased = batch + bias              # (10, 384) + (384,) → (10, 384)
print(f"After adding bias — first embedding, first 5 values:")
print(f"  before: {batch[0, :5].round(4)}")
print(f"  after:  {biased[0, :5].round(4)}")
print()

# --- Example 3: scale each embedding by a different weight ---
weights = np.array([0.5, 1.0, 2.0, 0.1, 0.8, 1.5, 0.3, 0.9, 1.2, 0.7])  # shape (10,)
weights_col = weights.reshape(10, 1)   # shape (10, 1) — needed to broadcast over dim-1
scaled = batch * weights_col           # (10, 384) * (10, 1) → (10, 384)
print(f"weights shape:     {weights.shape}")
print(f"weights_col shape: {weights_col.shape}")
print(f"scaled shape:      {scaled.shape}")
print("Norms before scaling:", np.round(np.linalg.norm(batch[:3], axis=1), 3))
print("Norms after scaling: ", np.round(np.linalg.norm(scaled[:3], axis=1), 3))

In [ ]:
# Visual: broadcasting a (3, 4) array with a (1, 4) row vector
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

A = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9,10,11,12]])
b = np.array([[10, 20, 30, 40]])
C = A + b

arr_specs = [
    (A, "A  shape (3, 4)",  '#4C9BE8', axes[0]),
    (b, "b  shape (1, 4)  →  broadcast to (3, 4)", '#E8A040', axes[1]),
    (C, "A + b  shape (3, 4)", '#5DBE7C', axes[2]),
]

for arr, title, color, ax in arr_specs:
    display = np.broadcast_to(arr, (3, 4))
    im = ax.imshow(display, cmap='Blues' if color == '#4C9BE8' else
                            ('Oranges' if color == '#E8A040' else 'Greens'),
                   vmin=0, vmax=display.max() + 1)
    for i in range(3):
        for j in range(4):
            ax.text(j, i, str(display[i, j]), ha='center', va='center',
                    fontsize=12, fontweight='bold', color='#222')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines[:].set_visible(False)

# Add plus and equals signs between subplots
fig.text(0.365, 0.52, '+', ha='center', va='center', fontsize=26, color='#555')
fig.text(0.635, 0.52, '=', ha='center', va='center', fontsize=26, color='#555')

fig.suptitle("Broadcasting: the (1, 4) row is stretched to match (3, 4)",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("In ML terms: the row b is like a bias vector added to every sample in a batch.")

---
## Part 5: The Dot Product and Matrix Multiply

These two operations are the backbone of everything — attention, cosine similarity, linear layers. Know them cold.

| Operation | Code | Result shape |
|-----------|------|--------------|
| 1D dot product (scalar) | `np.dot(a, b)` | scalar |
| Matrix × vector | `np.dot(A, v)` or `A @ v` | `(m,)` |
| Matrix × matrix | `np.dot(A, B)` or `A @ B` | `(m, p)` |

**Shape rule for `A @ B`:** the inner dimensions must match — `(m, n) @ (n, p) → (m, p)`.

Every time you see `@` in ML code, ask yourself: what are the shapes? What does the result mean?

In [ ]:
# 1. Dot product of two 1D vectors → scalar
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print(f"np.dot(a, b) = {np.dot(a, b)}   (= 1*4 + 2*5 + 3*6)")
print()

# 2. Matrix × vector → vector  (like a linear layer applied to one sample)
A = np.random.randn(128, 384)   # weight matrix: maps 384-dim input to 128-dim output
v = np.random.randn(384)         # one input embedding
result = A @ v
print(f"A.shape: {A.shape}")
print(f"v.shape: {v.shape}")
print(f"(A @ v).shape: {result.shape}   ← each row of A dot-products with v")
print()

# 3. Matrix × matrix  (entire batch through a linear layer at once)
batch = np.random.randn(10, 384)   # 10 input embeddings
result = batch @ A.T               # (10, 384) @ (384, 128) → (10, 128)
print(f"batch.shape: {batch.shape}")
print(f"A.T.shape:   {A.T.shape}")
print(f"(batch @ A.T).shape: {result.shape}   ← all 10 embeddings transformed at once")
print()
print("This is exactly what a nn.Linear layer does (minus the bias).")

In [ ]:
# The killer application: all pairwise similarity scores in ONE line
# This is what every production vector search does at its core.

np.random.seed(1)
queries    = np.random.randn(5, 384)    # 5 query embeddings
documents  = np.random.randn(20, 384)  # 20 stored document embeddings

# Normalize both so dot product == cosine similarity
queries   /= np.linalg.norm(queries,   axis=1, keepdims=True)
documents /= np.linalg.norm(documents, axis=1, keepdims=True)

# One line: (5, 384) @ (384, 20) → (5, 20)
# Entry [i, j] = cosine similarity between query i and document j
sim_matrix = queries @ documents.T

print(f"queries.shape:    {queries.shape}")
print(f"documents.T.shape: {documents.T.shape}")
print(f"sim_matrix.shape:  {sim_matrix.shape}")
print()
print("sim_matrix[i, j] = similarity between query i and document j")
print("For 5 queries × 20 docs: 100 similarities computed in one matrix multiply.")
print()
print("First query vs. all 20 docs:")
print(np.round(sim_matrix[0], 3))

### Exercise 2: All dot products in one line

You have a single query embedding `q` of shape `(384,)` and a matrix of 50 stored embeddings `E` of shape `(50, 384)`. Compute all 50 dot products — the similarity between `q` and every stored embedding — in one line, with no loop.

The result should have shape `(50,)`: one scalar per stored embedding.

**Hint:** Think about what `E @ q` gives you — check the shapes.

In [ ]:
np.random.seed(7)
q = np.random.randn(384)       # one query embedding, shape (384,)
E = np.random.randn(50, 384)   # 50 stored embeddings, shape (50, 384)

# YOUR TURN: compute all 50 dot products in one line (no loop).
# Hint: what shape does E @ q produce?
scores = ???  # ← replace this
raise NotImplementedError("Replace ??? with the matrix-vector multiply")

# ── Test ──────────────────────────────────────────────────────────────────────
assert scores.shape == (50,), f"Expected shape (50,), got {scores.shape}"
# Verify against a loop-based reference
reference = np.array([np.dot(E[i], q) for i in range(50)])
assert np.allclose(scores, reference, atol=1e-10), "Values don't match reference"
print(f"scores.shape: {scores.shape}")
print(f"scores[:5]: {np.round(scores[:5], 4)}")
print("✅ Passed! 50 dot products, zero loops.")

<details>
<summary>Solution (click to expand)</summary>

```python
scores = E @ q
```

Shape check: `(50, 384) @ (384,) → (50,)`. NumPy treats the 1D `q` as a column vector for the multiply, then returns a 1D result. This single matrix-vector multiply does the same work as 50 separate `np.dot` calls in a loop — but in one BLAS call, which is heavily optimized.

</details>

---
## Part 6: Normalization

Most embedding models output **unit-norm vectors**: their L2 norm is exactly 1.0. You saw this in the RAG notebook — `all-MiniLM-L6-v2` always outputs vectors with `np.linalg.norm(v) ≈ 1.0`.

Why does this matter?

- When vectors are unit-norm, **dot product equals cosine similarity** — no division needed. `np.dot(a, b)` is both fast and correct.
- When vectors are *not* unit-norm (some older models, raw logits), you must normalize before comparing — otherwise longer vectors dominate by magnitude, not meaning.

The normalization formula: `v_normalized = v / ||v||`

In NumPy: `v / np.linalg.norm(v)`

In [ ]:
# Single-vector normalization
v = np.array([3.0, 4.0])    # classic 3-4-5 right triangle
norm = np.linalg.norm(v)    # L2 norm = sqrt(3² + 4²) = 5.0
v_norm = v / norm

print(f"v:             {v}")
print(f"||v||:         {norm}  (= sqrt(3² + 4²))")
print(f"v_normalized:  {v_norm}")
print(f"||v_normalized||: {np.linalg.norm(v_norm):.10f}   ← exactly 1.0")
print()

# Verifying a real embedding model
# (This is what you'd see with sentence-transformers)
np.random.seed(42)
fake_embedding = np.random.randn(384) * 5    # unnormalized: norm ≠ 1
print(f"Unnormalized norm: {np.linalg.norm(fake_embedding):.4f}")
normalized = fake_embedding / np.linalg.norm(fake_embedding)
print(f"Normalized norm:   {np.linalg.norm(normalized):.10f}")

### Exercise 3: Normalize a batch of embeddings

Normalizing one vector is easy. The challenge is normalizing an entire batch efficiently — no loop, using broadcasting.

**Your task:** Complete `normalize_batch(E)` so that every row of the returned array has unit L2 norm.

**Key hint:** `np.linalg.norm(E, axis=1, keepdims=True)` returns the norm of each *row* as a column — shape `(n, 1)`. From there, broadcasting handles the rest.

In [ ]:
def normalize_batch(E: np.ndarray) -> np.ndarray:
    """
    Normalize each row of E to unit L2 norm.
    Input:  E of shape (n, d)
    Output: array of same shape where every row has ||row|| == 1.0
    """
    # YOUR TURN: two lines.
    # Line 1: compute the norm of each row, keeping dims so broadcasting works.
    # Line 2: divide E by those norms.
    raise NotImplementedError(
        "Hint: norms = np.linalg.norm(E, axis=1, keepdims=True), then return E / norms"
    )

# ── Test ──────────────────────────────────────────────────────────────────────
np.random.seed(3)
E = np.random.randn(20, 384) * np.random.rand(20, 1) * 10  # varying norms

E_norm = normalize_batch(E)

assert E_norm.shape == E.shape, f"Shape changed: {E.shape} → {E_norm.shape}"
row_norms = np.linalg.norm(E_norm, axis=1)
assert np.allclose(row_norms, 1.0, atol=1e-6), \
    f"Not all rows are unit norm: min={row_norms.min():.6f}, max={row_norms.max():.6f}"

print(f"Input norms  (first 5): {np.round(np.linalg.norm(E, axis=1)[:5], 3)}")
print(f"Output norms (first 5): {np.round(row_norms[:5], 6)}")
print(f"✅ Passed! All {E_norm.shape[0]} rows have unit norm.")

<details>
<summary>Solution (click to expand)</summary>

```python
def normalize_batch(E: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(E, axis=1, keepdims=True)
    return E / norms
```

`keepdims=True` is the key. Without it, `norms` has shape `(n,)` and the division fails (shape mismatch). With it, `norms` has shape `(n, 1)`, which broadcasts cleanly against `(n, d)` — each row is divided by its own norm.

This is exactly what `sentence-transformers` does internally when `normalize_embeddings=True`.

</details>

---
## Part 7: Key Operations for ML

A handful of functions show up constantly. Here they are in context.

In [ ]:
np.random.seed(11)
scores = np.array([0.12, 0.87, 0.34, 0.95, 0.61, 0.23, 0.78, 0.44])
docs   = [f"doc_{i}" for i in range(len(scores))]

print("Similarity scores:", np.round(scores, 2))
print()

# Reductions
print(f"np.mean(scores):   {np.mean(scores):.4f}   ← average score")
print(f"np.std(scores):    {np.std(scores):.4f}   ← spread")
print(f"np.sum(scores):    {np.sum(scores):.4f}")
print()

# Selection
print(f"np.argmax(scores): {np.argmax(scores)}         ← index of best match")
print(f"scores[argmax]:    {scores[np.argmax(scores)]}  ← its score")
print()

# Sorting (the critical one for retrieval)
sorted_idx = np.argsort(scores)            # ascending by default
print(f"np.argsort (ascending):  {sorted_idx}")
print(f"scores in that order:    {np.round(scores[sorted_idx], 2)}")
print()
top_idx = np.argsort(scores)[::-1]        # flip to descending
print(f"argsort descending:  {top_idx}")
print(f"scores descending:   {np.round(scores[top_idx], 2)}")
print(f"Ranked docs: {[docs[i] for i in top_idx]}")
print()

# Clipping — useful for logits, probabilities, normalized outputs
raw_logits = np.array([-2.5, 0.0, 0.3, 1.1, 3.7])
clipped    = np.clip(raw_logits, 0.0, 1.0)
print(f"np.clip(logits, 0, 1): {clipped}")

In [ ]:
# axis= is one of the most confusing things in NumPy — let's make it concrete.
np.random.seed(5)
batch = np.random.randn(4, 6)   # 4 embeddings, 6 dims each
print("batch (4 embeddings x 6 dims):")
print(np.round(batch, 2))
print()

# axis=0: collapse *across* rows → one value per column (per dimension)
mean_per_dim = np.mean(batch, axis=0)
print(f"np.mean(batch, axis=0) → shape {mean_per_dim.shape}   (mean value of each dimension)")
print(np.round(mean_per_dim, 3))
print()

# axis=1: collapse *across* columns → one value per row (per embedding)
mean_per_emb = np.mean(batch, axis=1)
print(f"np.mean(batch, axis=1) → shape {mean_per_emb.shape}   (mean value of each embedding)")
print(np.round(mean_per_emb, 3))
print()
print("Memory aid:")
print("  axis=0 → across rows → result has shape of one row")
print("  axis=1 → across cols → result has shape of one column")

---
## Exercise 4: Top-k Retrieval (Capstone)

This is the core of vector search. You have a query embedding and a matrix of stored embeddings. Return the indices and similarity scores of the k most similar stored embeddings — ranked, highest first.

**Your task:** Complete `top_k_similar` using only NumPy operations. No loop.

Steps:
1. Compute all dot products between `query` and each row of `embeddings` — shape `(n,)`
2. Use `np.argsort` to rank them — remember argsort is ascending by default
3. Return the top-k indices and their scores

Assume both `query` and the rows of `embeddings` are already unit-norm, so dot product == cosine similarity.

In [ ]:
def top_k_similar(query: np.ndarray, embeddings: np.ndarray, k: int):
    """
    Find the k most similar embeddings to query.

    Args:
        query:      shape (d,)    — a single unit-norm query vector
        embeddings: shape (n, d)  — n unit-norm stored vectors
        k:          number of results to return

    Returns:
        indices: shape (k,)  — indices of top-k embeddings, best first
        scores:  shape (k,)  — their similarity scores, descending
    """
    # YOUR TURN: ~3 lines.
    # 1. dot products
    # 2. argsort (flip to descending)
    # 3. slice top k, return indices and scores
    raise NotImplementedError(
        "Hint: scores = embeddings @ query; top_idx = np.argsort(scores)[::-1][:k]"
    )

# ── Test ──────────────────────────────────────────────────────────────────────
np.random.seed(99)
n, d, k = 100, 384, 5

query_raw = np.random.randn(d)
embeds_raw = np.random.randn(n, d)

# Normalize everything (unit-norm assumption)
query  = query_raw  / np.linalg.norm(query_raw)
embeds = embeds_raw / np.linalg.norm(embeds_raw, axis=1, keepdims=True)

indices, scores = top_k_similar(query, embeds, k)

assert indices.shape == (k,), f"Expected indices shape ({k},), got {indices.shape}"
assert scores.shape  == (k,), f"Expected scores shape ({k},), got {scores.shape}"
assert np.all(np.diff(scores) <= 0), "Scores must be sorted descending"
assert all(0 <= i < n for i in indices), "Indices must be valid"

# Verify they really are the top-k by brute force
all_scores = embeds @ query
expected_top = set(np.argsort(all_scores)[-k:])
assert set(indices) == expected_top, "Didn't return the actual top-k"

print(f"Top-{k} results:")
for rank, (idx, score) in enumerate(zip(indices, scores)):
    print(f"  [{rank+1}] embedding #{idx:3d}  score={score:.4f}")
print(f"✅ Passed! Correct top-{k} in descending order.")

<details>
<summary>Solution (click to expand)</summary>

```python
def top_k_similar(query: np.ndarray, embeddings: np.ndarray, k: int):
    scores  = embeddings @ query
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]
```

`np.argsort(scores)[::-1]` sorts all n indices from highest to lowest score, then `[:k]` takes the first k. `scores[top_idx]` uses fancy indexing to pull the corresponding scores.

This is the exact computation that every vector database — ChromaDB, Pinecone, pgvector — performs at its core. The differences are: (1) approximate search (HNSW, IVF) for large n, (2) persistence to disk, (3) metadata filtering. The core math is identical.

</details>

In [ ]:
# Visualize the similarity score distribution and top-k highlights
all_scores = embeds @ query
top_indices, top_scores = top_k_similar(query, embeds, k=5)

fig, ax = plt.subplots(figsize=(13, 4))

# All scores as a bar chart
bar_colors = ['#4C9BE8'] * n
for idx in top_indices:
    bar_colors[idx] = '#E8704C'

ax.bar(range(n), np.sort(all_scores)[::-1], color=sorted(
    zip(all_scores, bar_colors), reverse=True
)[0][1] if False else [
    '#E8704C' if i in set(top_indices) else '#4C9BE8'
    for i in np.argsort(all_scores)[::-1]
], width=0.8, edgecolor='none')

ax.axhline(top_scores[-1], color='#E8704C', lw=1.5, ls='--', alpha=0.7,
           label=f'Top-{k} threshold ({top_scores[-1]:.3f})')

ax.set_xlabel("Embedding rank (sorted by score)")
ax.set_ylabel("Cosine similarity")
ax.set_title(f"Top-{k} retrieval from {n} embeddings (orange = selected)",
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"Threshold score: {top_scores[-1]:.4f}")
print(f"Everything above that line gets retrieved.")

---
## Connecting Forward

Everything in the RAG notebook — embedding texts, building the vector store, computing cosine similarity, retrieving top-k — is built from exactly what you practiced here:

| RAG operation | NumPy underneath |
|---------------|------------------|
| `model.encode(texts)` | Returns `(n, 384)` array — shape |
| Cosine similarity | `np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))` |
| Retrieve top-k | `np.argsort(scores)[::-1][:k]` |
| Batch similarity | `queries @ documents.T` — matrix multiply |
| Normalize embeddings | `E / np.linalg.norm(E, axis=1, keepdims=True)` — broadcasting |

When you see this in real code:
```python
np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
```
you now know exactly what that's doing: project `a` onto `b`, normalized by their magnitudes so only the angle matters.

### Next: Notebook 2

**How LLMs actually see text — tokens, context windows, and your first API call.**

You'll learn: what a token is and why it matters for cost and context limits, how a tokenizer maps text to integers, what it means when a model "runs out of context", and how to make your first Claude API call.

---
*Notebook 1 of 5 — [ML Edge](https://mle-edge.dev) self-directed curriculum*